# Clase 3 — Práctico: crear un dataset y volver a cargarlo

**Del diccionario al archivo y del archivo al DataFrame: qué sobrevive al viaje**

Joselina Davyt-Colo — Facultad de Ciencias Empresariales y Economía — Universidad de Montevideo

---

En la clase 2 construimos un DataFrame a partir de la dinámica de la caja
negra. Ese DataFrame vivía en memoria: al cerrar la sesión desapareció.

Un dataset recién existe como objeto compartible cuando se escribe en un archivo. Y ese
archivo es, él mismo, un instrumento de medición: tiene una codificación, un separador,
una convención decimal y una forma de representar lo que falta. Si esas decisiones no se
registran, el archivo deja de ser un registro fiel del DGP y pasa a ser una fuente propia
de error.

Este práctico recorre ese viaje de ida y vuelta. Cada ejercicio está construido alrededor
de algo que falla; el objetivo no es evitar el error sino saber leerlo.

---

### Preparación

Antes de empezar, ejecutar una sola vez el generador de archivos:

    py generar_archivos.py          (Windows)
    python3 generar_archivos.py     (Mac / Linux)

Crea la carpeta `datos/` con cuatro archivos. Tres están deliberadamente mal formados.

## Ejercicio 1. Del diccionario al DataFrame: el tipo es una decisión de medición

*Declarar explícitamente el tipo de cada variable y justificarlo.*

1. Reconstruir el diccionario `datos_caja` de la clase 2 y pasarlo a DataFrame.
2. Declarar `textura` como categórica nominal y `rigidez` como categórica **ordenada** (`blando < medio < rígido`).
3. Declarar `certeza_observador` como categórica ordenada de 1 a 5.
4. Verificar con `.dtypes` que ninguna columna quedó como texto genérico.
5. Responder en la bitácora: en `num_aristas`, un objeto esférico tiene `0`. ¿Ese cero es una medición o es un «no aplica»? ¿Cambia la respuesta si el observador no supo contar las aristas?

In [ ]:
import pandas as pd
import numpy as np

datos_caja = {
    "objeto_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "textura": ["suave", "rugoso", "poroso", "suave", "rugoso",
                "suave", "poroso", "rugoso", "suave", "poroso"],
    "rigidez": ["blando", "rígido", "medio", "blando", "rígido",
                "rígido", "blando", "medio", "medio", "blando"],
    "num_aristas": [0, 8, 0, 4, 12, 0, 0, 6, 4, 0],
    "peso_estimado_g": [45.0, 120.5, 15.0, 85.0, 210.0,
                        35.0, 18.5, 95.0, 70.0, 22.0],
    "certeza_observador": [5, 4, 3, 5, 4, 2, 4, 3, 4, 3],
}

df = pd.DataFrame(datos_caja)

# TODO 1: declarar textura como categórica nominal

# TODO 2: declarar rigidez como categórica ordenada

# TODO 3: declarar certeza_observador como categórica ordenada de 1 a 5

print(df.dtypes)

## Ejercicio 2. Las restricciones del DGP, escritas como código

*Convertir supuestos implícitos sobre la medición en verificaciones ejecutables.*

1. Escribir cuatro verificaciones que codifiquen restricciones del proceso de medición: el peso es estrictamente positivo; la certeza está entre 1 y 5; `num_aristas` es un entero no negativo; `objeto_id` no tiene repetidos.
2. Envolverlas en una función `validar(df)` que devuelva la lista de problemas encontrados en lugar de detenerse en el primero.
3. Probarla con el DataFrame correcto y después con `df_roto`, que tiene tres errores introducidos a propósito.

In [ ]:
df_roto = df.copy()
df_roto.loc[0, "peso_estimado_g"] = -12.0
df_roto.loc[3, "num_aristas"] = -1
df_roto.loc[7, "objeto_id"] = 1


def validar(datos):
    """Devuelve una lista de problemas. Lista vacía = dataset válido."""
    problemas = []

    # TODO 1: peso estrictamente positivo

    # TODO 2: certeza entre 1 y 5

    # TODO 3: num_aristas entero no negativo

    # TODO 4: objeto_id sin repetidos

    return problemas


print("df       ->", validar(df))
print("df_roto  ->", validar(df_roto))

## Ejercicio 3. El diccionario de variables

*Documentar el instrumento de medición junto con el dato.*

1. Construir un segundo DataFrame, `diccionario`, con una fila por variable y las columnas: `variable`, `tipo`, `unidad`, `instrumento`, `rango_valido`, `faltante`.
2. En `instrumento` describir cómo se obtuvo el valor (por ejemplo: «estimación táctil del observador, 15 segundos, sin ver el objeto»).
3. Guardarlo como `datos/diccionario_variables.csv`.
4. Responder en la bitácora: ¿qué columna del diccionario habría evitado el problema del `0` en `num_aristas` del ejercicio 1?

In [ ]:
from pathlib import Path

CARPETA = Path("datos")
CARPETA.mkdir(exist_ok=True)

# TODO: completar una fila por cada variable del dataset
diccionario = pd.DataFrame([
    {
        "variable": "peso_estimado_g",
        "tipo": "numérica continua",
        "unidad": "gramos",
        "instrumento": "estimación táctil del observador, 15 s, sin ver el objeto",
        "rango_valido": "> 0",
        "faltante": "vacío",
    },
    # ...
])

diccionario.to_csv(CARPETA / "diccionario_variables.csv",
                   index=False, encoding="utf-8")
print(diccionario)

## Ejercicio 4. El viaje de ida y vuelta: qué se pierde al guardar

*Comprobar que el CSV no conserva los tipos declarados.*

1. Guardar `df` como `datos/caja_negra_mio.csv` **sin** el argumento `index=False`.
2. Volver a leerlo y comparar `.dtypes` y `.columns` con los del DataFrame original.
3. Identificar las dos pérdidas: una columna que apareció y un tipo que se degradó.
4. Corregir la escritura y reconstruir el tipo ordenado en la lectura.
5. Repetir el mismo viaje con `.to_parquet()` / `.read_parquet()` y comparar el resultado (requiere `pip install pyarrow`).

In [ ]:
# --- ida ---
df.to_csv("datos/caja_negra_mio.csv")          # falta un argumento

# --- vuelta ---
df_leido = pd.read_csv("datos/caja_negra_mio.csv")

print("columnas originales:", list(df.columns))
print("columnas releídas  :", list(df_leido.columns))
print()
print("rigidez original :", df["rigidez"].dtype)
print("rigidez releída  :", df_leido["rigidez"].dtype)

# TODO 1: reescribir el CSV sin la columna de índice

# TODO 2: reconstruir rigidez como categórica ordenada después de leer

# TODO 3: hacer el mismo viaje con parquet y comparar

## Ejercicio 5. El CSV que armó Excel

*Diagnosticar y corregir separador, decimal y codificación.*

1. Intentar leer `datos/caja_negra_excel.csv` con `pd.read_csv()` sin argumentos. Copiar el mensaje de error completo en la bitácora.
2. Abrir el archivo con un editor de texto (no con Excel) y describir qué se ve: ¿cuál es el separador de columnas? ¿y el decimal?
3. Leerlo correctamente usando `sep`, `decimal` y `encoding`.
4. Verificar que `peso_estimado_g` quedó numérica y no texto.
5. Responder: si se hubiera leído con `sep=';'` pero sin `decimal=','`, ¿el código habría fallado o habría seguido adelante con datos mal tipados?

In [ ]:
RUTA = "datos/caja_negra_excel.csv"

# --- intento ingenuo: registrar el error ---
try:
    malo = pd.read_csv(RUTA)
    print(malo.head())
except Exception as e:
    print(type(e).__name__, ":", e)

# TODO 1: leer el archivo con los tres argumentos correctos
bueno = pd.read_csv(RUTA)      # completar

# TODO 2: verificar que el peso quedó numérico

# TODO 3: probar qué pasa si se omite decimal=","

## Ejercicio 6. Dónde está el archivo

*Usar rutas que funcionen en la máquina de otra persona.*

1. Imprimir el directorio de trabajo actual con `Path.cwd()`.
2. Verificar con `.exists()` que el archivo está donde se lo busca, **antes** de intentar leerlo, y emitir un mensaje claro si no está.
3. Reescribir la ruta de manera que el script funcione sin importar desde qué carpeta se ejecute.
4. Responder: ¿por qué una ruta como `C:/Users/jodavyt/Documents/GitHub/.../caja_negra.csv` rompe el trabajo en equipo?

In [ ]:
from pathlib import Path

print("Directorio de trabajo:", Path.cwd())

ruta = Path("datos") / "caja_negra.csv"

# TODO 1: verificar que existe antes de leer, con un mensaje útil si no está

# TODO 2: construir la ruta de forma independiente del directorio de trabajo
#         (en un script: Path(__file__).parent; en un notebook: Path.cwd())

df_base = pd.read_csv(ruta)
print(df_base.shape)

## Ejercicio 7. El -99 que se lleva puesta la media

*Detectar faltantes disfrazados de dato válido.*

1. Leer `datos/caja_negra_faltantes.csv` sin argumentos especiales y calcular la media de `peso_estimado_g`.
2. Inspeccionar los valores únicos de cada columna y encontrar las **cuatro** maneras distintas en que ese archivo codifica un faltante: `-99`, `s/d`, `NA` y celda vacía.
3. Determinar cuáles de esas cuatro reconoce `read_csv()` por sí solo y cuáles no.
4. Volver a leer declarándolas con `na_values` y recalcular la media.
5. Reportar `.isna().sum()` por columna.
6. Responder: la diferencia entre ambas medias, ¿es un problema de programación o un problema de medición?

In [ ]:
RUTA = "datos/caja_negra_faltantes.csv"

ingenuo = pd.read_csv(RUTA)
print("Media ingenua:", round(ingenuo["peso_estimado_g"].mean(), 2))
print()

# TODO 1: inspeccionar los valores únicos de cada columna
#         (usar pd.isna() para no mezclar faltantes con texto al ordenar)
for col in ingenuo.columns:
    pass

# TODO 2: releer declarando los cuatro códigos de faltante
declarado = pd.read_csv(RUTA)       # completar con na_values=...

# TODO 3: comparar medias y contar faltantes por columna

## Ejercicio 8. Lectura a ciegas del archivo del equipo vecino

*Inspeccionar un archivo desconocido y reconstruir su DGP.*

1. Cargar `datos/equipo_vecino.csv`, un registro transaccional sin documentación.
2. Ejecutar la inspección mínima: `.shape`, `.dtypes`, `.head()`, `.isna().sum()`, `.describe(include='all')`.
3. Comparar la columna de monto en el archivo de texto con lo que quedó en el DataFrame. El archivo dice `1.295`; ¿qué valor cargó pandas? ¿Y para `980`?
4. Corregir cuatro problemas: nombres de columna con espacios y mayúsculas; la fecha leída como texto; el separador de miles del monto; los espacios sobrantes en `sucursal`, que hacen que `Centro` y `Centro ` cuenten como categorías distintas.
5. Escribir en la bitácora tres afirmaciones sobre el proceso que generó ese archivo, y una pregunta que habría que hacerle al equipo que lo produjo.

In [ ]:
RUTA = "datos/equipo_vecino.csv"

# --- lectura ingenua ---
vecino = pd.read_csv(RUTA, encoding="utf-8")
print(vecino.shape)
print(vecino.dtypes, "\n")
print(vecino.head())

# Comparar con el texto crudo del archivo:
print("\nPrimeras lineas del archivo tal cual:")
for linea in open(RUTA, encoding="utf-8").read().splitlines()[:4]:
    print("   ", linea)

# TODO 1: releer declarando el separador de miles (argumento thousands)

# TODO 2: convertir la fecha, que viene como dd/mm/aaaa

# TODO 3: normalizar los nombres de columna (minúsculas, sin espacios)

# TODO 4: limpiar los espacios sobrantes en sucursal y pasarla a categórica

# TODO 5: verificar con .dtypes, .describe() y el listado de categorías

**Desafío por equipo (entrega en la bitácora).**

Aplicar el recorrido completo a la estructura de datos del propio equipo: construir un
dataset mínimo de diez observaciones con la variable central de esa estructura, escribir
su diccionario de variables, guardarlo en `datos/`, volver a cargarlo en una sesión nueva
y verificar que nada se perdió en el viaje.

Cada estructura tiene un punto de quiebre distinto en este recorrido; identificarlo es
parte de la consigna.

| Equipo | Punto de quiebre esperado |
|---|---|
| Series macroeconómicas | La frecuencia y el índice temporal no sobreviven al CSV |
| Microdatos de encuesta | El ponderador y los códigos de no respuesta |
| Sistemas transaccionales | El identificador, los duplicados y la zona horaria |
| Datos experimentales | La asignación a tratamiento y el momento de medición |
| Datos espaciales | El sistema de coordenadas de referencia |
| Texto a red | La correspondencia entre nodos y aristas, y la codificación |
| Uso de IA | La marca de tiempo, la sesión y la unidad de observación |


---

**Criterio de corrección.** Un ejercicio está resuelto cuando otra persona puede ejecutar
el archivo en su máquina, sin editar ninguna ruta, y obtener exactamente el mismo resultado.
No alcanza con que funcione en la computadora de quien lo escribió.